# 01 Unstructured Multi-Tenant RAG with Custom Scopes

## Overview

In this tutorial we'll build a multi-tenant RAG (Retrieval-Augmented Generation) system using OAuth 2.0 Access Tokens with custom scopes. This pattern is essential for SaaS applications where multiple tenants share the same infrastructure but need strict data isolation.

We'll use Cognito custom scopes to identify insurance tiers (regular vs platinum), then apply metadata filtering at the Knowledge Base level to ensure each tier only sees their own documents. This is the production-recommended approach for multi-tenant applications.

In this example, we'll:
1. Create a Cognito User Pool with custom resource server and scopes
2. Build an OpenSearch Serverless collection and Knowledge Base
3. Upload tier-specific documents with metadata
4. Build a Strands Agent that filters by tier using Access Token scopes
5. Test multi-tier isolation

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Multi-Tenant RAG |
| Authentication | OAuth 2.0 Access Token with custom scopes |
| Agentic Framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4.5 |
| Tenant isolation | Metadata filtering |
| Tutorial vertical | Insurance |
| Example complexity | Intermediate |
| SDK used | Amazon Bedrock AgentCore Python SDK |

### Tutorial Architecture

```
Regular User → Access Token (scope: insurance/regular) → AgentCore → Strands Agent
                                                                           ↓
                                                             RetrieveAndGenerate API
                                                             + filter: {tier: regular}
                                                                           ↓
                                                             Knowledge Base (pooled)
                                                             ├─ Regular Policy {tier: regular} ✓
                                                             ├─ Platinum Policy {tier: platinum} ✗
                                                             └─ Regular Claims {tier: regular} ✓
```

### Tutorial Key Features

* Access Tokens with custom scopes (`insurance/regular`, `insurance/platinum`)
* Metadata-based tier isolation at vector DB layer
* Pooled Knowledge Base architecture (cost-efficient)
* Document-level access control by insurance tier
* Production-ready pattern using OAuth 2.0 standards

## Prerequisites

To execute this tutorial you will need:
* Python 3.9+
* AWS credentials configured
* Amazon Bedrock access
* Bedrock AgentCore SDK
* Strands Agents

In [ ]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands-agents pyyaml

In [ ]:
import sys
import os

# Add parent directory to path for utils
sys.path.insert(0, os.path.abspath('..'))

import boto3
import json
import time
import yaml
from boto3.session import Session
from utils.aws_resources import (
    CognitoManager,
    S3Manager,
    KnowledgeBaseManager,
    IAMManager
)

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print("\n✓ AWS utilities loaded")

## Configuration

We'll use different resource names from Tutorial 01 for the Cognito User Pool (since we need custom attributes), but we'll reuse the same S3 bucket and Knowledge Base.

In [ ]:
PROJECT_NAME = "multitenant-rag"

user_pool_name = f"{PROJECT_NAME}-pool"
bucket_name = f"{PROJECT_NAME}-kb-{account_id}"
kb_name = f"{PROJECT_NAME}-kb"
kb_role_name = f"{PROJECT_NAME}-kb-role"
collection_name = f"{PROJECT_NAME}-collection"

print("Configuration:")
print(f"  User Pool: {user_pool_name}")
print(f"  S3 Bucket: {bucket_name}")
print(f"  Knowledge Base: {kb_name}")
print(f"  Collection: {collection_name}")

## Setting Up Multi-Tier Cognito with Custom Scopes

For multi-tier insurance applications, we'll use Cognito custom scopes to identify user tiers. This is the production-recommended approach - instead of putting tier information in custom attributes, we define scopes like `insurance/regular` and `insurance/platinum` that get included in the Access Token.

When users authenticate, their Access Token will contain a scope claim (e.g., `insurance/regular`) which our agent can read and use for filtering documents by insurance tier.

In [ ]:
print("Setting up Cognito User Pool...\n")

cognito = CognitoManager(region)
cognito_client = boto3.client('cognito-idp', region_name=region)

# Get or create user pool
cognito_config = cognito.get_or_create_user_pool(
    pool_name=user_pool_name,
    schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True}
    ],
    auto_verified_attributes=['email']
)

user_pool_id = cognito_config['user_pool_id']
client_id = cognito_config['client_id']
discovery_url = cognito_config['discovery_url']

print(f"✓ User Pool ID: {user_pool_id}")
print(f"✓ Client ID: {client_id}")

# Create resource server with custom scopes
resource_server_id = "insurance"
try:
    cognito_client.describe_resource_server(
        UserPoolId=user_pool_id,
        Identifier=resource_server_id
    )
    print(f"✓ Resource server exists: {resource_server_id}")
except cognito_client.exceptions.ResourceNotFoundException:
    cognito_client.create_resource_server(
        UserPoolId=user_pool_id,
        Identifier=resource_server_id,
        Name="Insurance API",
        Scopes=[
            {'ScopeName': 'regular', 'ScopeDescription': 'Regular tier access'},
            {'ScopeName': 'platinum', 'ScopeDescription': 'Platinum tier access'}
        ]
    )
    print(f"✓ Created resource server with scopes: insurance/regular, insurance/platinum")

# Update app client to include custom scopes
# Note: We only need USER_PASSWORD_AUTH, not OAuth flows
try:
    cognito_client.update_user_pool_client(
        UserPoolId=user_pool_id,
        ClientId=client_id,
        ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH']
    )
    print(f"✓ App client configured for USER_PASSWORD_AUTH")
    print(f"  Note: Custom scopes (insurance/regular, insurance/platinum) are available via resource server")
except Exception as e:
    print(f"Note: {e}")

### Creating Multi-Tier Test Users

Let's create test users for two different insurance tiers. We'll assign each user to a Cognito group (regular or platinum), which will determine which scope they get in their Access Token. In a real application, users would be assigned to groups based on their insurance plan.

In [ ]:
test_users = [
    {
        'username': 'john-regular',
        'password': 'Pass123!',
        'email': 'john.regular@example.com',
        'tier': 'regular'
    },
    {
        'username': 'sarah-platinum',
        'password': 'Pass123!',
        'email': 'sarah.platinum@example.com',
        'tier': 'platinum'
    }
]

# Create groups for each tier
for tier in ['regular', 'platinum']:
    try:
        cognito_client.create_group(
            GroupName=tier,
            UserPoolId=user_pool_id,
            Description=f"{tier.capitalize()} tier insurance users"
        )
        print(f"  Created group: {tier}")
    except cognito_client.exceptions.GroupExistsException:
        print(f"  Group exists: {tier}")

# Create users and assign to groups
for user in test_users:
    cognito.get_or_create_user(
        user_pool_id=user_pool_id,
        username=user['username'],
        password=user['password'],
        attributes=[
            {'Name': 'email', 'Value': user['email']},
            {'Name': 'email_verified', 'Value': 'true'}
        ]
    )
    
    # Add user to tier group
    try:
        cognito_client.admin_add_user_to_group(
            UserPoolId=user_pool_id,
            Username=user['username'],
            GroupName=user['tier']
        )
    except:
        pass
    
    print(f"  User ready: {user['username']} → {user['tier']} tier")

print(f"\n✓ {len(test_users)} users ready with tier assignments")

## Setting Up S3 and Tier-Specific Documents

Now we'll create an S3 bucket and add documents for each insurance tier. The key here is the metadata files - each document gets a `.metadata.json` file that contains the `tier`. This metadata is indexed by the Knowledge Base and used for filtering.

In [ ]:
s3 = S3Manager(region)
bucket_name = s3.get_or_create_bucket(bucket_name)

print(f"✓ S3 bucket ready: {bucket_name}")

### Uploading Tier Documents with Metadata

Each document needs a companion `.metadata.json` file. This is how Bedrock Knowledge Bases understand document metadata for filtering. The metadata file contains the `tier` attribute that matches our custom scopes.

In [ ]:
s3_client = boto3.client('s3', region_name=region)

# Sample documents for each insurance tier
documents = {
    'regular': {
        'policy.txt': 'Regular Tier Insurance Policy: Maximum deductible for auto insurance is $1000. Coverage includes collision and comprehensive. Standard roadside assistance available during business hours.',
        'claims.txt': 'Regular Tier Claims: Standard processing time of 5-7 business days. Claims filed in 2024: 5 claims. Average claim amount: $2500. Online claims portal available.'
    },
    'platinum': {
        'policy.txt': 'Platinum Tier Insurance Policy: Maximum deductible for auto insurance is $500. Premium coverage includes collision, comprehensive, rental car reimbursement, and 24/7 concierge roadside assistance.',
        'claims.txt': 'Platinum Tier Claims: Priority processing within 24-48 hours. Claims filed in 2024: 2 claims. Average claim amount: $1800. Dedicated claims specialist and expedited service.'
    }
}

uploaded_count = 0
for tier, docs in documents.items():
    for filename, content in docs.items():
        # Upload document
        s3_key = f"tiers/{tier}/{filename}"
        
        s3_client.put_object(Bucket=bucket_name, Key=s3_key, Body=content)
        print(f"  Uploaded: {s3_key}")
        
        # Upload metadata file
        metadata = {
            "metadataAttributes": {
                "tier": tier,
                "document_type": filename.split('.')[0]
            }
        }
        metadata_key = f"{s3_key}.metadata.json"
        
        try:
            s3_client.head_object(Bucket=bucket_name, Key=metadata_key)
        except:
            s3_client.put_object(
                Bucket=bucket_name,
                Key=metadata_key,
                Body=json.dumps(metadata)
            )
            print(f"  Metadata: {metadata_key}")
        
        uploaded_count += 1

print(f"\n✓ {uploaded_count} tier-specific documents ready with metadata")

## Creating the Knowledge Base Infrastructure

Now we'll create the complete Knowledge Base infrastructure: IAM role, OpenSearch Serverless collection, vector index, and the Knowledge Base itself.

In [ ]:
from utils.aws_resources import OpenSearchServerlessManager

# Create IAM role for Knowledge Base
iam = IAMManager()

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

kb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{bucket_name}/*",
                f"arn:aws:s3:::{bucket_name}"
            ]
        },
        {
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["aoss:APIAccessAll"],
            "Resource": f"arn:aws:aoss:{region}:{account_id}:collection/*"
        }
    ]
}

kb_role_arn = iam.get_or_create_role(
    role_name=kb_role_name,
    assume_role_policy=trust_policy,
    policies={'KBPolicy': kb_policy}
)

print(f"✓ IAM Role: {kb_role_arn}")
print("  Waiting for IAM propagation...")
time.sleep(10)

# Create OpenSearch Serverless collection
oss = OpenSearchServerlessManager(region)

collection_config = oss.get_or_create_collection(
    collection_name=collection_name,
    kb_role_arn=kb_role_arn
)

collection_id = collection_config['collection_id']
collection_arn = collection_config['collection_arn']
collection_endpoint = collection_config['collection_endpoint']

print(f"✓ Collection: {collection_endpoint}")

# Create vector index
index_name = oss.create_vector_index(
    collection_endpoint=collection_endpoint,
    index_name='bedrock-kb-index'
)

print(f"✓ Vector index: {index_name}")

# Create Knowledge Base
kb_manager = KnowledgeBaseManager(region)

knowledge_base_id = kb_manager.get_or_create_knowledge_base(
    kb_name=kb_name,
    kb_role_arn=kb_role_arn,
    collection_arn=collection_arn,
    index_name=index_name
)

print(f"✓ Knowledge Base: {knowledge_base_id}")

### Ingesting Documents

Now we'll connect our S3 bucket as a data source and start the ingestion job. This will index all our tier-specific documents with their metadata.

In [ ]:
# Create data source
data_source_id = kb_manager.create_data_source(
    kb_id=knowledge_base_id,
    bucket_name=bucket_name
)

print(f"✓ Data source: {data_source_id}")

# Start ingestion
print("\nStarting ingestion...")
kb_manager.sync_data_source(
    kb_id=knowledge_base_id,
    data_source_id=data_source_id
)
print("✓ Knowledge Base ready with indexed documents")

## Creating the Multi-Tier Agent

Now we'll create a Strands Agent that understands tier-based isolation using Access Token scopes. The agent extracts the tier from the `scope` claim in the Access Token (e.g., `insurance/regular`) and uses it to filter Knowledge Base queries.

The agent code below will be saved to `rag_metadata_agent.py` and includes a `retrieve_and_generate` tool that applies metadata filtering by insurance tier.

In [ ]:
%%writefile rag_metadata_agent.py

from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import json
import boto3

app = BedrockAgentCoreApp()

model = BedrockModel(
    model_id="global.anthropic.claude-sonnet-4-5-20250929-v1:0"
)

bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')

@tool
def retrieve_and_generate(query: str, knowledge_base_id: str, tier: str) -> str:
    """
    Retrieve information from the knowledge base with tier filtering.
    
    Args:
        query: The user's question
        knowledge_base_id: The ID of the knowledge base to query
        tier: The insurance tier for filtering documents (regular or platinum)
    
    Returns:
        Generated response based on retrieved documents for the tier
    """
    try:
        # Build metadata filter for tier isolation
        retrieval_filter = {
            "equals": {
                "key": "tier",
                "value": tier
            }
        }
        
        response = bedrock_agent_runtime.retrieve_and_generate(
            input={'text': query},
            retrieveAndGenerateConfiguration={
                'type': 'KNOWLEDGE_BASE',
                'knowledgeBaseConfiguration': {
                    'knowledgeBaseId': knowledge_base_id,
                    'modelArn': 'arn:aws:bedrock:us-west-2::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0',
                    'retrievalConfiguration': {
                        'vectorSearchConfiguration': {
                            'filter': retrieval_filter
                        }
                    }
                }
            }
        )
        
        output_text = response['output']['text']
        citations = response.get('citations', [])
        
        result = {
            'answer': output_text,
            'citations_count': len(citations),
            'has_sources': len(citations) > 0,
            'tier': tier,
            'filter_applied': retrieval_filter
        }
        
        return json.dumps(result)
        
    except Exception as e:
        return json.dumps({
            'error': str(e),
            'answer': 'I encountered an error accessing the knowledge base.',
            'citations_count': 0,
            'has_sources': False,
            'tier': tier
        })

agent = Agent(
    model=model,
    system_prompt="You are a helpful AI assistant with access to a multi-tier insurance knowledge base. When answering questions, use the retrieve_and_generate tool to search the knowledge base with tier filtering.",
    tools=[retrieve_and_generate]
)

@app.entrypoint
def invoke_agent(payload):
    user_input = payload.get("prompt", "")
    user_identity = payload.get("user_identity", {})
    kb_id = payload.get("knowledge_base_id")
    
    # Extract tier from Access Token scope
    scope = user_identity.get('scope', '')
    tier = None
    
    # Parse scope to find insurance tier (e.g., \"insurance/regular\")
    if 'insurance/regular' in scope:
        tier = 'regular'
    elif 'insurance/platinum' in scope:
        tier = 'platinum'
    
    print(f"User: {user_identity.get('username', 'unknown')}")
    print(f"Scope: {scope}")
    print(f"Tier: {tier}")
    print(f"Query: {user_input}")
    
    # If knowledge base ID is provided, include tier filtering
    if kb_id and tier:
        enhanced_input = f"{user_input}\n\nKnowledge Base ID: {kb_id}\nTier: {tier}"
        response = agent(enhanced_input)
    else:
        response = agent(user_input)
    
    return {
        "response": response.message['content'][0]['text'],
        "user_identity": {
            "sub": user_identity.get('sub'),
            "username": user_identity.get('username'),
            "scope": scope,
            "tier": tier,
            "authenticated": user_identity.get('authenticated', False)
        },
        "knowledge_base_used": kb_id is not None,
        "tier_filter_applied": kb_id is not None and tier is not None
    }

if __name__ == "__main__":
    app.run()

### Deploying to AgentCore Runtime

We'll deploy this agent with Access Token authentication. AgentCore Runtime will validate the JWT and extract the scope claim, which our agent uses to determine the user's insurance tier.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

print("Configuring AgentCore Runtime...\n")

runtime = Runtime()

runtime.configure(
    entrypoint="rag_metadata_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="rag_metadata",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedAudience": [client_id]
        }
    }
)

# Add Authorization header forwarding
with open('.bedrock_agentcore.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['agents']['rag_metadata']['request_header_configuration'] = {
    'requestHeaderAllowlist': ['Authorization']
}

with open('.bedrock_agentcore.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✓ Agent configured")

In [ ]:
print("Deploying agent...\n")
launch_result = runtime.launch(auto_update_on_conflict=True)
print(f"\n✓ Deployed: {launch_result.agent_arn}")

In [ ]:
print("Waiting for agent...")
status_response = runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED']:
    time.sleep(10)
    status_response = runtime.status()
    status = status_response.endpoint['status']

print(f"✓ Agent ready: {status}")

## Testing Multi-Tenant Isolation

Now for the exciting part! We'll test that each tenant can only see their own documents. We'll ask the same question to both tenants and verify they get different answers based on their tenant-specific data.

In [ ]:
def get_tokens(username, password, scope='openid email insurance/regular'):
    cognito_client = boto3.client('cognito-idp', region_name=region)
    response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    return response['AuthenticationResult']

print("Authenticating users...\n")
regular_tokens = get_tokens('john-regular', 'Pass123!', 'openid email insurance/regular')
platinum_tokens = get_tokens('sarah-platinum', 'Pass123!', 'openid email insurance/platinum')
print("✓ Users authenticated with tier-specific scopes")

In [ ]:
print("💼 TEST: Regular Tier - Auto Insurance Deductible")
print("="*60)

# Use Access Token (contains custom scopes)
access_token_regular = regular_tokens['AccessToken']

response_regular = runtime.invoke(
    {
        "prompt": "What is the maximum deductible for auto insurance?",
        "knowledge_base_id": knowledge_base_id
    },
    bearer_token=access_token_regular
)

resp_data = response_regular.get('response')
if isinstance(resp_data, str):
    try:
        parsed = json.loads(resp_data)
    except:
        parsed = {'response': resp_data, 'user_identity': {}}
else:
    parsed = resp_data

print(f"\nResponse: {parsed.get('response', 'N/A')}")
print(f"\nUser Identity:")
user_id = parsed.get('user_identity', {})
print(f"  Tier: {user_id.get('tier')}")
print(f"  Username: {user_id.get('username')}")
print(f"\nTier Filter Applied: {parsed.get('tier_filter_applied', False)}")

In [ ]:
print("\n💎 TEST: Platinum Tier - Auto Insurance Deductible")
print("="*60)

# Use Access Token (contains custom scopes)
access_token_platinum = platinum_tokens['AccessToken']

response_platinum = runtime.invoke(
    {
        "prompt": "What is the maximum deductible for auto insurance?",
        "knowledge_base_id": knowledge_base_id
    },
    bearer_token=access_token_platinum
)

resp_data = response_platinum.get('response')
if isinstance(resp_data, str):
    try:
        parsed = json.loads(resp_data)
    except:
        parsed = {'response': resp_data, 'user_identity': {}}
else:
    parsed = resp_data

print(f"\nResponse: {parsed.get('response', 'N/A')}")
print(f"\nUser Identity:")
user_id = parsed.get('user_identity', {})
print(f"  Tier: {user_id.get('tier')}")
print(f"  Username: {user_id.get('username')}")
print(f"\nTier Filter Applied: {parsed.get('tier_filter_applied', False)}")

print("\n" + "="*60)
print("✓ Multi-tier isolation verified!")
print("  Regular tier sees: $1000 deductible (standard coverage)")
print("  Platinum tier sees: $500 deductible (premium coverage)")
print("  Same query, different results based on tier filtering")

## Summary

In this tutorial, we:

1. Created a Cognito User Pool with custom resource server and scopes (`insurance/regular`, `insurance/platinum`)
2. Built complete Knowledge Base infrastructure (S3, OpenSearch Serverless, IAM roles)
3. Added tier-specific documents (regular vs platinum) with metadata files
4. Built a Strands Agent that extracts tier from Access Token scopes
5. Deployed to AgentCore Runtime with Access Token authentication
6. Verified multi-tier isolation with the same query returning different results

### Key Takeaways

- **Access Tokens with custom scopes** - production-recommended OAuth 2.0 pattern
- **Metadata filtering** provides tier isolation at the vector DB layer
- **Pooled architecture** is cost-efficient - one KB serves all tiers
- **Defense-in-depth** - multiple layers of security (JWT validation + scope checking + metadata filtering)
- **Standards-compliant** - follows OAuth 2.0 best practices

### Next Steps

- **Tutorial 02**: Row-Level Security for structured data with PostgreSQL RLS
- **Tutorial 03**: Build a complete multi-tier platform with supervisor agent and UI

## Cleanup (Optional)

Run this cell to delete all AWS resources created in this tutorial.

**Warning**: This will permanently delete:
- Knowledge Base and data sources
- OpenSearch Serverless collection and policies
- S3 bucket and all documents
- IAM role and policies
- Cognito User Pool and users

In [ ]:
from utils.aws_resources import cleanup_knowledge_base_resources, cleanup_cognito_resources

# Uncomment to run cleanup
# cleanup_knowledge_base_resources(
#     region=region,
#     kb_name=kb_name,
#     collection_name=collection_name,
#     bucket_name=bucket_name,
#     role_name=kb_role_name
# )

# cleanup_cognito_resources(
#     region=region,
#     user_pool_name=user_pool_name
# )

# Congratulations!

You've successfully built a multi-tenant RAG system with metadata filtering!